In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

try:
    from prophet import Prophet
except ImportError:
    raise ImportError("Run: pip install prophet")

In [9]:
DATA_PATH = r"C:\Users\kaija\Downloads\Nat_Gas.csv"

df = pd.read_csv(DATA_PATH)
df.columns = ["date", "price"]
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%y")
df = df.sort_values("date").reset_index(drop=True)

df_prophet = df.rename(columns={"date": "ds", "price": "y"})

FORECAST_MONTHS = 36

ml_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive",
    interval_width=0.95,
    changepoint_prior_scale=0.05,
)
ml_model.fit(df_prophet)

future = ml_model.make_future_dataframe(periods=FORECAST_MONTHS, freq="MS")
forecast = ml_model.predict(future)
hist_end = df["date"].iloc[-1]
forecast_end = forecast["ds"].iloc[-1]

def get_ml_price(date_input):
    if isinstance(date_input, str):
        date_input = pd.to_datetime(date_input)
    if date_input < df["date"].iloc[0]:
        raise ValueError(f"{date_input.date()} is before the training data start.")
    if date_input > forecast_end:
        raise ValueError(f"{date_input.date()} is beyond the forecast horizon ({forecast_end.date()}).")
    pred = ml_model.predict(pd.DataFrame({"ds": [date_input]}))
    return (
        round(float(pred["yhat"].iloc[0]), 4),
        round(float(pred["yhat_lower"].iloc[0]), 4),
        round(float(pred["yhat_upper"].iloc[0]), 4),
    )

print(f"Prophet model trained on {len(df)} monthly data points.")
print(f"Historical data : {df['date'].iloc[0].date()} to {hist_end.date()}")
print(f"Forecast window : up to {forecast_end.date()}")

20:54:49 - cmdstanpy - INFO - Chain [1] start processing
20:54:50 - cmdstanpy - INFO - Chain [1] done processing


Prophet model trained on 48 monthly data points.
Historical data : 2020-10-31 to 2024-09-30
Forecast window : up to 2027-09-01


In [10]:
def price_storage_contract(
    injection_date,
    withdrawal_date,
    max_volume_mmbtu,
    injection_cost_per_mmbtu=0.05,
    withdrawal_cost_per_mmbtu=0.05,
    storage_cost_per_mmbtu_month=0.02,
    discount_rate=0.05,
    current_date=None,
):
    if current_date is None:
        current_date = pd.Timestamp.today().normalize()

    inj_date  = pd.to_datetime(injection_date)
    with_date = pd.to_datetime(withdrawal_date)

    if inj_date >= with_date:
        raise ValueError("Injection date must be before withdrawal date.")

    inj_price,  inj_low,  inj_high  = get_ml_price(inj_date)
    with_price, with_low, with_high = get_ml_price(with_date)

    storage_months = max(
        (with_date.year - inj_date.year) * 12 + (with_date.month - inj_date.month), 1
    )

    total_cost = (
        injection_cost_per_mmbtu
        + withdrawal_cost_per_mmbtu
        + storage_cost_per_mmbtu_month * storage_months
    )

    spread = with_price - inj_price - total_cost

    t_years = max((with_date - pd.to_datetime(current_date)).days / 365.25, 0)
    discount_factor = np.exp(-discount_rate * t_years)

    contract_value = spread * max_volume_mmbtu * discount_factor

    spread_bull = with_high - inj_low  - total_cost
    spread_bear = with_low  - inj_high - total_cost
    value_bull  = spread_bull * max_volume_mmbtu * discount_factor
    value_bear  = spread_bear * max_volume_mmbtu * discount_factor

    return {
        "injection_date"        : inj_date.date(),
        "withdrawal_date"       : with_date.date(),
        "injection_price"       : inj_price,
        "withdrawal_price"      : with_price,
        "storage_months"        : storage_months,
        "injection_cost"        : round(injection_cost_per_mmbtu, 4),
        "withdrawal_cost"       : round(withdrawal_cost_per_mmbtu, 4),
        "storage_cost_total"    : round(storage_cost_per_mmbtu_month * storage_months, 4),
        "total_cost_per_mmbtu"  : round(total_cost, 4),
        "spread_per_mmbtu"      : round(spread, 4),
        "volume_mmbtu"          : max_volume_mmbtu,
        "discount_rate"         : discount_rate,
        "discount_factor"       : round(discount_factor, 4),
        "contract_value"        : round(contract_value, 2),
        "value_bull_95"         : round(value_bull, 2),
        "value_bear_95"         : round(value_bear, 2),
    }

In [11]:
style  = {"description_width": "210px"}
layout = widgets.Layout(width="440px")

inj_date_w   = widgets.Text(value="2025-04-01", description="Injection date (YYYY-MM-DD)",  style=style, layout=layout)
with_date_w  = widgets.Text(value="2025-11-01", description="Withdrawal date (YYYY-MM-DD)", style=style, layout=layout)
volume_w     = widgets.FloatText(value=100000,  description="Volume (MMBtu)",               style=style, layout=layout, step=1000)
inj_cost_w   = widgets.FloatText(value=0.05,   description="Injection cost ($/MMBtu)",     style=style, layout=layout, step=0.01)
with_cost_w  = widgets.FloatText(value=0.05,   description="Withdrawal cost ($/MMBtu)",    style=style, layout=layout, step=0.01)
store_cost_w = widgets.FloatText(value=0.02,   description="Storage $/MMBtu/month",        style=style, layout=layout, step=0.005)
discount_w   = widgets.FloatText(value=5.0,    description="Discount rate (%)",            style=style, layout=layout, step=0.5)

calc_btn = widgets.Button(
    description="Calculate Contract Value",
    button_style="primary",
    layout=widgets.Layout(width="220px", margin="12px 0 0 0"),
)
out = widgets.Output()

def on_calculate(b):
    out.clear_output()
    with out:
        try:
            r = price_storage_contract(
                injection_date=inj_date_w.value.strip(),
                withdrawal_date=with_date_w.value.strip(),
                max_volume_mmbtu=volume_w.value,
                injection_cost_per_mmbtu=inj_cost_w.value,
                withdrawal_cost_per_mmbtu=with_cost_w.value,
                storage_cost_per_mmbtu_month=store_cost_w.value,
                discount_rate=discount_w.value / 100.0,
            )
            print("=" * 52)
            print("  GAS STORAGE CONTRACT VALUATION")
            print("=" * 52)
            print(f"  Injection date       : {r['injection_date']}")
            print(f"  Withdrawal date      : {r['withdrawal_date']}")
            print(f"  Storage period       : {r['storage_months']} month(s)")
            print(f"  Volume               : {r['volume_mmbtu']:>12,.0f} MMBtu")
            print()
            print("  -- Price Forecast (Prophet ML) --")
            print(f"  Injection price      : ${r['injection_price']:>9.4f} /MMBtu")
            print(f"  Withdrawal price     : ${r['withdrawal_price']:>9.4f} /MMBtu")
            print()
            print("  -- Cost Breakdown --")
            print(f"  Injection cost       : ${r['injection_cost']:>9.4f} /MMBtu")
            print(f"  Withdrawal cost      : ${r['withdrawal_cost']:>9.4f} /MMBtu")
            print(f"  Storage cost (total) : ${r['storage_cost_total']:>9.4f} /MMBtu")
            print(f"  Total costs          : ${r['total_cost_per_mmbtu']:>9.4f} /MMBtu")
            print()
            print("  -- Valuation --")
            print(f"  Net spread           : ${r['spread_per_mmbtu']:>9.4f} /MMBtu")
            print(f"  Discount rate        : {r['discount_rate']*100:.2f}%")
            print(f"  Discount factor      : {r['discount_factor']:.4f}")
            print()
            val = r['contract_value']
            print(f"  CONTRACT VALUE       : ${val:>12,.2f}")
            print(f"  Bull case (95% CI)   : ${r['value_bull_95']:>12,.2f}")
            print(f"  Bear case (95% CI)   : ${r['value_bear_95']:>12,.2f}")
            print("=" * 52)
            if val > 0:
                print("  VERDICT: Positive value -- spread covers all costs.")
            else:
                print("  VERDICT: Negative value -- costs exceed the spread.")
        except Exception as e:
            print(f"  Error: {e}")

calc_btn.on_click(on_calculate)

form = widgets.VBox([
    widgets.HTML("<b>Dates</b>"),
    inj_date_w, with_date_w,
    widgets.HTML("<b>Contract Terms</b>"),
    volume_w, inj_cost_w, with_cost_w, store_cost_w,
    widgets.HTML("<b>Financials</b>"),
    discount_w,
    calc_btn,
])

display(widgets.HBox([form, out]))